In [1]:
import pandas
from numpy import poly1d, polyfit
from src.energiapy.components.temporal_scale import Temporal_scale
from src.energiapy.components.resource import Resource
from src.energiapy.components.process import Process
from src.energiapy.components.material import Material
from src.energiapy.components.location import Location
from src.energiapy.components.network import Network
from src.energiapy.components.scenario import Scenario
from src.energiapy.components.transport import Transport
from src.energiapy.components.result import Result
from src.energiapy.model.formulate_milp import formulate_milp
from src.energiapy.utils.data_utils import get_data, make_henry_price_df
from src.energiapy.utils.nsrdb_utils import fetch_nsrdb_data
from src.energiapy.plot import plot
from src.energiapy.model.pyomo_solve import solve
from src.energiapy.utils.cluster_utils import reduce_scenario, agg_hierarchial_elbow, Clustermethod
from src.energiapy.utils.data_utils import load_results
from src.energiapy.model.pyomo_sets import generate_sets
from src.energiapy.model.pyomo_vars import *
from src.energiapy.model.pyomo_cons import *
from src.energiapy.model.pyomo_objs import cost_objective, uncertainty_cost_objective, demand_objective
import matplotlib.pyplot as plt
from pyomo.environ import ConcreteModel, Suffix
import random
from matplotlib import rc

In [2]:
weather20_df = pandas.read_csv('data/ho_solar20.csv', index_col=0)
weather20_df.index = [i.split('+')[0] for i in weather20_df.index]
weather = weather20_df[~weather20_df.index.str.contains('02-29')] #remove leap years
weather

,wind_speed,dni
2020-01-01 00:00:00,9.5,0.0
2020-01-01 01:00:00,7.5,0.0
2020-01-01 02:00:00,6.0,0.0
2020-01-01 03:00:00,6.0,0.0
2020-01-01 04:00:00,6.0,0.0
...,...,...
2020-12-31 19:00:00,54.5,0.5
2020-12-31 20:00:00,55.5,0.0
2020-12-31 21:00:00,50.0,0.0
2020-12-31 22:00:00,46.0,181.5


In [3]:
ercot20 = pandas.read_excel('data/Native_Load_2020.xlsx')
ercot = ercot20[['COAST']]
ercot['index'] = weather20_df.index
ercot = ercot.set_index('index')
ercot = ercot[~ercot.index.str.contains('02-29')]
ercot

C:\Users\cacodcar\AppData\Local\Temp\ipykernel_12516\4195252575.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ercot['index'] = weather20_df.index


,COAST
index,
2020-01-01 00:00:00,9890.091522
2020-01-01 01:00:00,9751.586415
2020-01-01 02:00:00,9603.421087
2020-01-01 03:00:00,9501.585599
2020-01-01 04:00:00,9499.426925
...,...
2020-12-31 19:00:00,12068.071946
2020-12-31 20:00:00,11818.903690
2020-12-31 21:00:00,11560.408167


In [4]:
ng_price20 = make_henry_price_df(
    file_name='data/Henry_Hub_Natural_Gas_Spot_Price_Daily.csv', year=2020, stretch=True)
ng_price_df = ng_price20
ng_price_df['index'] = weather.index
ng_price_df = ng_price_df.drop(columns= 'scales')
ng_price_stretch = ng_price_df.set_index('index')

In [5]:
cost_dict = get_data(file_name='data/cost_dict')
for i in cost_dict['HO']['moderate'].keys():
    print(i + ':', cost_dict['HO']['moderate'][i]['0'])

LiI_c: {'CAPEX': 1302181.81818181, 'Fixed O&M': 41432.7272727272, 'Variable O&M': 0, 'units': '$/MW', 'source': 'NREL Annual Technology Baseline 2021, https://atb.nrel.gov/'}
LiI_d: {'CAPEX': 0.001, 'Fixed O&M': 0.001, 'Variable O&M': 0, 'units': '$/MW', 'source': 'Dummy Process'}
CAES_c: {'CAPEX': 1669000.0, 'Fixed O&M': 16700.0, 'Variable O&M': 0, 'units': '$/MW', 'source': 'https://www.pnm.com/documents/396023/1506047/2017+-+HDR+10-30-17+PNM+Energy+Storage+Report.pdf/a2b7ca65-e1ba-92c8-308a-9a8391a87331'}
CAES_d: {'CAPEX': 0.001, 'Fixed O&M': 0.001, 'Variable O&M': 0, 'units': '$/MW', 'source': 'Dummy Process'}
PSH_c: {'CAPEX': 0, 'Fixed O&M': 41432.7272727272, 'Variable O&M': 4435.188, 'units': '$/MW', 'source': 'NREL Annual Technology Baseline 2021, https://atb.nrel.gov/'}
PSH_d: {'CAPEX': 0.001, 'Fixed O&M': 0.001, 'Variable O&M': 0, 'units': '$/MW', 'source': 'Dummy Process'}
PV: {'CAPEX': 1302181.81818181, 'Fixed O&M': 41432.7272727272, 'Variable O&M': 0, 'units': '$/MW', 'sour

In [6]:
bigM = 10**3  # very large number
smallM = 0.1
water_price = 31.70  # $/5000gallons
power_price = 8  # cents/kWh
ur_price = 42.70  # 250 Pfund U308 (Uranium)
A_f = 0.05  # annualization factor
# CO2_res = 0.2
pv_start = 0
ake_start = 0
smrh_start = 0
smr_start = 0
asmr_start = 0


In [7]:
Charge = Resource(name='Charge', sell=False,
                  store_max=bigM, basis='MW', label='Battery energy', block='energystorage')
Solar = Resource(
    name='Solar', cons_max=bigM, basis='MW', label='Solar Power', block='energyfeedstock')
Wind = Resource(name='Wind', cons_max= bigM, basis='MW', label='Wind Power', block='energyfeedstock')
H2_L = Resource(name='H2_L', store_max=10**4, revenue=2,
                mile=1/(0.1180535*1.60934), basis='kg', label='Hydrogen - Geological', block='resourcestorage')
H2 = Resource(name='H2', basis='kg', sell = True, demand = True, label='Hydrogen', block='Resource')
H2O = Resource(name='H2O', cons_max=10**6,
               price= 0.001, basis='kg', label='Water', block='Resource')
            #    price=water_price/(5000*3.7854), basis='kg', label='Water', block='Resource')
O2 = Resource(name='O2', sell=True, loss=0.07,
              basis='kg', label='Oxygen', block='Resource')
CH4 = Resource(name='CH4', cons_max=10 **
               6, price=1, basis='kg', label='Natural gas', block='materialfeedstock')
CO2 = Resource(name='CO2', basis='kg',
               label='Carbon dioxide', block='Resource')
CO2_Vent = Resource(
    name='CO2_Vent', sell=True, basis='kg', label='Carbon dioxide - Vented', block='resourcedischarge')
# Power= Resource(name= 'Power', sell= True, store_max=0,   \
#    mile= (10**3)/(0.2167432**1.60934), label= 'Renewable power generated')
Power = Resource(name='Power', basis='MW',
                 label='Renewable power generated', block='Resource')

In [8]:
LiI_c = Process(name='LiI_c', conversion={Charge: 1, Power: -1}, cost = cost_dict['HO']['moderate']['LiI_c']['0'],\
    prod_max=5, trl='nrel', block='power_storage', label='Lithium-ion battery', citation='Zakeri 2015')
LiI_d = Process(name='LiI_d', conversion={Charge: -1.1765, Power: 1}, cost =  {'CAPEX': smallM, 'Fixed O&M': 0, 'Variable O&M': smallM, \
    'units': '$/kg','source': 'dummy'}, \
    prod_max=5, trl='discharge', block='power_storage', label='Lithium-ion battery discharge', citation='Zakeri 2015')
WF = Process(name='WF', conversion={Wind: -1, Power: 1, H2O: -1}, cost=cost_dict['HO']['moderate']['WF']['0'],
             prod_max=100, gwp=52700, land=10800/1800, trl='nrel', block='power_generation',
             label='Wind mill array', citation='Use windtoolkit conversion', p_fail= 0.05)
PV = Process(name='PV', intro_scale=pv_start, conversion={Solar: -1, Power: 1, H2O: -20}, cost=cost_dict['HO']['moderate']['PV']['0'],
             prod_max=bigM, gwp=53000, land=13320/1800, trl='nrel', block='power_generation', \
                 label='Solar photovoltaics (PV) array', citation='Use pvlib conversion', p_fail = 0.05)
AKE = Process(name='AKE', intro_scale=ake_start, conversion={Power: -1, H2: 19.474, O2: 763.2, H2O: -175.266},
              cost=cost_dict['HO']['moderate']['AKE']['0'], prod_max=bigM, trl='utility', block='material_production',
              label='Alkaline water electrolysis (AWE)', citation='Demirhan et al. 2018 AIChE paper', p_fail= 0.05)  # 20.833 MW required to produce 1000t/day.H2
SMR = Process(name='SMR', intro_scale=smr_start, cost= {'CAPEX': 2400, 'Fixed O&M': 800, 'Variable O&M': 0.05, 'units': '$/kg', 'source': 'dummy'}, \
    conversion={Power: -1.11*10**(-3), CH4: -3.76, H2O: -23.7, H2: 1, CO2_Vent: 9.4979}, prod_max=bigM, gwp=0, trl='enterprise',
                      block='material_production', label='Steam methane reforming', citation='Mosca 2020', p_fail= 0.05)
H2_L_c = Process(name='H2_L_c', conversion={Power: -4.17*10**(-4), H2_L: 1, H2: -1}, cost={'CAPEX': 1000, 'Fixed O&M': 600, 'Variable O&M': 0.05,
                                                                                           'units': '$/kg', 'source': 'dummy'},
                 prod_max=bigM, gwp=0, trl='repurposed', block='material_storage', label='Hydrogen geological storage',
                 citation='Bossel and Eliasson - Energy and the Hydrogen Economy', p_fail= 0.05)
H2_L_d = Process(name='H2_L_d', conversion={H2_L: -1, H2: 1}, prod_max=bigM, gwp=0, trl='nocost', cost={'CAPEX': smallM, 'Fixed O&M': 0, 'Variable O&M': 1,                                                                                                        'units': '$/kg', 'source': 'dummy'},
                 block='material_storage', label='Hydrogen geological storage discharge', citation='Bossel and Eliasson - Energy and the Hydrogen Economy', p_fail= 0.01)


In [9]:
scales = Temporal_scale(discretization_list=[1, 365, 24])

In [10]:
HO = Location(name='HO', processes={LiI_c, LiI_d, PV, WF, AKE, SMR, H2_L_c, H2_L_d}, scales=scales, label='Houston', \
    demand_factor= {H2: ercot}, cost_factor = {CH4: ng_price_stretch}, \
        capacity_factor = {PV: pandas.DataFrame(weather['dni']), WF: pandas.DataFrame(weather['wind_speed'])}\
            , demand_level=2, capacity_level= 2, cost_level= 2)

In [11]:
scenario = Scenario(name= 'shell', network= HO, scales= scales,  expenditure_scale_level= 2, scheduling_scale_level= 2, \
    network_scale_level= 0, demand_scale_level= 2, label= 'shell milp case study (HO)', )

In [22]:

def flexibility_MIP_reformulation(scenario: Scenario, affix_results: Result = None, carbon_bound:float= None, \
    carbon_reduction_percentage:float= 0.0, penalty:float = 0.0, demand:float= 0.0):
    instance = ConcreteModel()

    generate_sets(instance=instance, location_set=scenario.location_set, transport_set=scenario.transport_set, scales=scenario.scales,
                  process_set=scenario.process_set, resource_set=scenario.resource_set, material_set=scenario.material_set,
                  source_set=scenario.source_locations, sink_set=scenario.sink_locations)

    generate_scheduling_vars(
        instance=instance, scale_level=scenario.scheduling_scale_level)
    generate_network_vars(
        instance=instance, scale_level=scenario.network_scale_level)
    
    generate_network_binary_vars(
        instance=instance, scale_level=scenario.network_scale_level)

    generate_uncertainty_vars(
        instance=instance, scale_level=scenario.demand_scale_level)

    if len(instance.locations) > 1:
        generate_transport_vars(
            instance=instance, scale_level=scenario.scheduling_scale_level)

    inventory_balance_constraint(instance=instance, scheduling_scale_level=scenario.scheduling_scale_level,
                                 conversion=scenario.conversion)
    nameplate_production_constraint(instance=instance, capacity_factor=scenario.capacity_factor,
                                    network_scale_level=scenario.network_scale_level, scheduling_scale_level=scenario.scheduling_scale_level)
    
    nameplate_inventory_constraint(instance=instance, loc_res_dict=scenario.loc_res_dict, network_scale_level=scenario.network_scale_level,
                                   scheduling_scale_level=scenario.scheduling_scale_level)
    resource_consumption_constraint(instance=instance, loc_res_dict=scenario.loc_res_dict,
                                    cons_max=scenario.cons_max, scheduling_scale_level=scenario.scheduling_scale_level)
    resource_purchase_constraint(instance=instance, cost_factor=scenario.cost_factor, price=scenario.price,
                                 loc_res_dict=scenario.loc_res_dict, scheduling_scale_level=scenario.scheduling_scale_level,
                                 expenditure_scale_level=scenario.expenditure_scale_level)
    
    nameplate_production_failure_constraint(instance=instance, fail_factor=scenario.fail_factor,
                                    network_scale_level=scenario.network_scale_level, scheduling_scale_level=scenario.scheduling_scale_level)
    
    # production_facility_affix_constraint(instance=instance, affix_production_cap = affix_results.output['Cap_P'],
    #                                loc_pro_dict=scenario.loc_pro_dict, network_scale_level=scenario.network_scale_level)
    
    # storage_facility_affix_constraint(instance=instance, affix_storage_cap = affix_results.output['Cap_S'],
    #                             loc_res_dict=scenario.loc_res_dict, network_scale_level=scenario.network_scale_level)

    production_facility_constraint(instance=instance, prod_max=scenario.prod_max,
                                   loc_pro_dict=scenario.loc_pro_dict, network_scale_level=scenario.network_scale_level)
    storage_facility_constraint(instance=instance, store_max=scenario.store_max,
                                loc_res_dict=scenario.loc_res_dict, network_scale_level=scenario.network_scale_level)
    
    location_production_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level, cluster_wt=scenario.cluster_wt)
    location_discharge_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level, cluster_wt=scenario.cluster_wt)
    location_consumption_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level, cluster_wt=scenario.cluster_wt)
    location_purchase_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level, cluster_wt=scenario.cluster_wt)

    network_production_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_discharge_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_consumption_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_purchase_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)

    process_capex_constraint(instance=instance, capex_dict=scenario.capex_dict,
                             network_scale_level=scenario.network_scale_level)
    process_fopex_constraint(instance=instance, fopex_dict=scenario.fopex_dict,
                             network_scale_level=scenario.network_scale_level)
    process_vopex_constraint(instance=instance, vopex_dict=scenario.vopex_dict,
                             network_scale_level=scenario.network_scale_level)

    process_land_constraint(instance=instance, land_dict=scenario.land_dict,
                            network_scale_level=scenario.network_scale_level)
    location_land_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_land_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)

    location_capex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    location_fopex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    location_vopex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)

    network_capex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_fopex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_vopex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)


    demand_constraint_flex(instance=instance, demand_scale_level=scenario.demand_scale_level,
                      scheduling_scale_level=scenario.scheduling_scale_level, demand = demand, \
                          demand_factor=scenario.demand_factor)
    
    if carbon_bound is not None:
        carbon_emission_constraint(instance= instance, network_scale_level= scenario.network_scale_level, \
            carbon_reduction_percentage = carbon_reduction_percentage, carbon_bound = carbon_bound)
    
    instance.dual = Suffix(direction=Suffix.IMPORT)


    uncertainty_cost_objective(instance=instance,
                   network_scale_level=scenario.network_scale_level, uncertainty_scale_level =\
                       scenario.demand_scale_level, penalty = penalty)
    
    
    # demand_objective(instance=instance, network_scale_level =scenario.network_scale_level)
    
    

    return instance

In [23]:
reduced_scenario, rep_days_stretch,  info =  reduce_scenario(
    scenario=scenario, location= HO, periods=40, scale_level=1, method=Clustermethod.agg_hierarchial,\
        include = ['demand', 'capacity', 'cost'])

In [24]:
flex_MILP = flexibility_MIP_reformulation(scenario= reduced_scenario, penalty =  1, demand = 10**7)

In [15]:
results_reduced = solve(scenario = reduced_scenario, instance= flex_MILP, solver= 'gurobi', \
    name=f"mickey_mouse (reduced)", print_solversteps = True)


Set parameter QCPDual to value 1
Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 35608 rows, 21220 columns and 97849 nonzeros
Model fingerprint: 0x9259ce72
Variable types: 21210 continuous, 10 integer (10 binary)
Coefficient statistics:
  Matrix range     [4e-04, 1e+06]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+03, 1e+07]
Presolve removed 24394 rows and 11869 columns
Presolve time: 0.06s
Presolved: 11214 rows, 9351 columns, 32495 nonzeros
Variable types: 9351 continuous, 0 integer (0 binary)

Root relaxation: objective 7.109625e+09, 14 iterations, 0.01 seconds (0.01 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    7.109625e+09 7.1096e+09  0.00%     -    0s

Explored 1 nodes (14 simplex iter

In [34]:

for i, j in product([1, 4, 5, 6, 8], [100000, 200000, 300000, 400000 ]):

    flex_MILP = flexibility_MIP_reformulation(scenario= reduced_scenario, penalty =  i, demand = j)
    results_reduced = solve(scenario = reduced_scenario, instance= flex_MILP, solver= 'gurobi', \
        name=f"mickey_mouse (reduced)", print_solversteps = False)
    
    print(i, j, results_reduced.output['objective'])


1 100000 58347595.384959556
1 200000 116695190.76991911
1 300000 175042786.15487853
1 400000 233390381.53983822
4 100000 233390381.53983822
4 200000 466780763.07967645
4 300000 700171144.6195141
4 400000 933561526.1593529
5 100000 291737976.9247978
5 200000 583475953.8495955
5 300000 875213930.7743926
5 400000 1166951907.699191
6 100000 350085572.30975735


0.33333333333142906

In [18]:

def MILP_demand(scenario: Scenario, affix_results: Result = None, carbon_bound:float= None, \
    carbon_reduction_percentage:float= 0.0, penalty:float = 0.0, demand:float= 0.0):
    instance = ConcreteModel()

    generate_sets(instance=instance, location_set=scenario.location_set, transport_set=scenario.transport_set, scales=scenario.scales,
                  process_set=scenario.process_set, resource_set=scenario.resource_set, material_set=scenario.material_set,
                  source_set=scenario.source_locations, sink_set=scenario.sink_locations)

    generate_scheduling_vars(
        instance=instance, scale_level=scenario.scheduling_scale_level)
    generate_network_vars(
        instance=instance, scale_level=scenario.network_scale_level)
    
    generate_network_binary_vars(
        instance=instance, scale_level=scenario.network_scale_level)

    generate_uncertainty_vars(
        instance=instance, scale_level=scenario.demand_scale_level)

    if len(instance.locations) > 1:
        generate_transport_vars(
            instance=instance, scale_level=scenario.scheduling_scale_level)

    inventory_balance_constraint(instance=instance, scheduling_scale_level=scenario.scheduling_scale_level,
                                 conversion=scenario.conversion)
    nameplate_production_constraint(instance=instance, capacity_factor=scenario.capacity_factor,
                                    network_scale_level=scenario.network_scale_level, scheduling_scale_level=scenario.scheduling_scale_level)
    
    nameplate_inventory_constraint(instance=instance, loc_res_dict=scenario.loc_res_dict, network_scale_level=scenario.network_scale_level,
                                   scheduling_scale_level=scenario.scheduling_scale_level)
    resource_consumption_constraint(instance=instance, loc_res_dict=scenario.loc_res_dict,
                                    cons_max=scenario.cons_max, scheduling_scale_level=scenario.scheduling_scale_level)
    resource_purchase_constraint(instance=instance, cost_factor=scenario.cost_factor, price=scenario.price,
                                 loc_res_dict=scenario.loc_res_dict, scheduling_scale_level=scenario.scheduling_scale_level,
                                 expenditure_scale_level=scenario.expenditure_scale_level)
    
    nameplate_production_failure_constraint(instance=instance, fail_factor=scenario.fail_factor,
                                    network_scale_level=scenario.network_scale_level, scheduling_scale_level=scenario.scheduling_scale_level)
    
    # production_facility_affix_constraint(instance=instance, affix_production_cap = affix_results.output['Cap_P'],
    #                                loc_pro_dict=scenario.loc_pro_dict, network_scale_level=scenario.network_scale_level)
    
    # storage_facility_affix_constraint(instance=instance, affix_storage_cap = affix_results.output['Cap_S'],
    #                             loc_res_dict=scenario.loc_res_dict, network_scale_level=scenario.network_scale_level)

    production_facility_constraint(instance=instance, prod_max=scenario.prod_max,
                                   loc_pro_dict=scenario.loc_pro_dict, network_scale_level=scenario.network_scale_level)
    storage_facility_constraint(instance=instance, store_max=scenario.store_max,
                                loc_res_dict=scenario.loc_res_dict, network_scale_level=scenario.network_scale_level)
    
    location_production_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level, cluster_wt=scenario.cluster_wt)
    location_discharge_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level, cluster_wt=scenario.cluster_wt)
    location_consumption_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level, cluster_wt=scenario.cluster_wt)
    location_purchase_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level, cluster_wt=scenario.cluster_wt)

    network_production_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_discharge_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_consumption_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_purchase_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)

    process_capex_constraint(instance=instance, capex_dict=scenario.capex_dict,
                             network_scale_level=scenario.network_scale_level)
    process_fopex_constraint(instance=instance, fopex_dict=scenario.fopex_dict,
                             network_scale_level=scenario.network_scale_level)
    process_vopex_constraint(instance=instance, vopex_dict=scenario.vopex_dict,
                             network_scale_level=scenario.network_scale_level)

    process_land_constraint(instance=instance, land_dict=scenario.land_dict,
                            network_scale_level=scenario.network_scale_level)
    location_land_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_land_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)

    location_capex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    location_fopex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    location_vopex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)

    network_capex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_fopex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)
    network_vopex_constraint(
        instance=instance, network_scale_level=scenario.network_scale_level)


    demand_constraint_flex(instance=instance, demand_scale_level=scenario.demand_scale_level,
                      scheduling_scale_level=scenario.scheduling_scale_level, demand = 4.87398e+08/40, \
                          demand_factor=scenario.demand_factor)
    
    if carbon_bound is not None:
        carbon_emission_constraint(instance= instance, network_scale_level= scenario.network_scale_level, \
            carbon_reduction_percentage = carbon_reduction_percentage, carbon_bound = carbon_bound)
    
    instance.dual = Suffix(direction=Suffix.IMPORT)

    
    demand_objective(instance=instance, network_scale_level =scenario.network_scale_level)
    
    

    return instance

In [20]:
flex_MILP = MILP_demand(scenario= reduced_scenario)
results_reduced = solve(scenario = reduced_scenario, instance= flex_MILP, solver= 'gurobi', \
    name=f"mickey_mouse (reduced)", print_solversteps = True)


Set parameter QCPDual to value 1
Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 35608 rows, 21220 columns and 97849 nonzeros
Model fingerprint: 0xfd6fc90a
Variable types: 21210 continuous, 10 integer (10 binary)
Coefficient statistics:
  Matrix range     [4e-04, 1e+06]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+03, 1e+07]
Presolve removed 33597 rows and 17097 columns
Presolve time: 0.04s
Presolved: 2011 rows, 4123 columns, 6387 nonzeros
Variable types: 4123 continuous, 0 integer (0 binary)

Root relaxation: objective 6.446593e+09, 677 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    6.446593e+09 6.4466e+09  0.00%     -    0s

Explored 1 nodes (677 simplex iter